# The Annotated Triple Product Property Matrix Multiplication Algorithm by [Murage Kibicho](https://x.com/murage_kibicho)

<div align="center">
<img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/DuaLipa.png?raw=true:" alt="Dua Lipa alongside TPP paper" width=800>
</div>
I'm going to show you how to perform linear algebra without matrices.
Microsoft and Caltech researchers (Cohn & Umans, 2003) found an FFT-style matmul algo but it's a theoretical computer science artifact.

 It's built on **four observations**:
1. Wedderburn's theorem tells us there's a one-to-one map between the product of square matrices and the group algebra of a finite group.
<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/Wedderburn.png?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>

2. This map is a Discrete Fourier Transform.
3. Every Discrete Fourier Transform (DFT) admits a Fast Fourier Transform (FFT).
4. So there might exist an FFT-style algorithm for sub-quadratic (not even sub-cubic lol!) matrix multiplications.


This guide is best followed (in a separate tab) alongside this [LeetArxiv article](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication). You'll see the references and coding steps.





---

###### Trump defunded my math PhD and my realtor day job contract ends on 10th July.

If you’re hiring please reach out via email: murage@kibicho.com. If not, please consider supporting my [Substack](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).

---
**I must warn you. No one’s coded this paper before, so even ChatGPT has nothing to parrot. Please let me know if you spot an extreme misinterpretation. I could be wrong about everything lol.**

## Abstract Group Class
If you know nothing about mathematical groups like the alternating group or symmetric group then we give a terse, but thorough intro in [Section 1.1](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).



We code an abstract `Group` class that can accommodate different groups cyclic groups over the integers, or whatever you want.
<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/f43ebc14-2432-44e0-9bb8-53782fec3a6e_1614x864.jpg?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>

---


### Triple Product Property
This is [Section 2.2](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).

Informally, the *Triple Product Property* helps us identify mathematical groups where we can map our matmuls into FFT-style convolutions.

More formally (Cohn & Umans, 2003), a group $G$ realizes $\langle n_1,n_2,n_3\rangle$ if there are subsets $S_1,S_2,S_3 \subseteq G$ such that $|S_i| = n_i$ and for $q_i \in Q(S_i)$, if
$q_1q_2q_3 = 1$
then $q_1 = q_2 = q_3 = 1$.

We call this condition on $S_1,S_2,S_3$ the *triple product property*. If we wish to emphasize the specific subsets, we say that $G$ realizes $\langle n_1,n_2,n_3\rangle$ through $S_1,S_2,S_3$.


(Hedtke & Murthy, 2012) show us how to test for TPP using the right quotient. We use their $S, T, U$ variables in place of (Cohn & Umans, 2003) $S_1,S_2,S_3$ notation.

Here's a summary for the mathematically-inclined:
<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/9099655f-cc1a-4306-b89b-97dc08a6b7f1_1338x920.jpg?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>


In [1]:
import itertools
from typing import Set, Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
from abc import ABC, abstractmethod
import numpy as np
import time
class Group(ABC):
  #Abstract base class for any finite group
  @abstractmethod
  def elements(self) -> List:
    """Return all group elements"""
    pass
  @abstractmethod
  def identity(self):
    """Return the identity element"""
    pass
  @abstractmethod
  def groupOperation(self, a, b):
    """Group Operation for two elements: a op b = c """
    pass
  @abstractmethod
  def inverse(self, a):
    """Return inverse of element a"""
    pass
  @abstractmethod
  def elementToString(self, elem) -> str:
    """Convert element to string representation"""
    pass

  def Q(self, X: Set) -> Set:
    """Compute right quotient Q(X) = {x * y^(-1) : x, y in X}"""
    result = set()
    for x in X:
      for y in X:
        result.add(self.groupOperation(x, self.inverse(y)))
    return result

  def TestTPPNaive(self, S: Set, T: Set, U: Set) -> bool:
    """We follow Algorithm 5.2 from (Hedtke & Murthy, 2012)"""
    e = self.identity()
    QS = self.Q(S)
    QT = self.Q(T)
    QU = self.Q(U)

    for s in QS:
      for t in QT:
        for u in QU:
          if self.groupOperation(self.groupOperation(s, t), u) == e:
            if s != e or t != e or u != e:
              return False
    return True


  def BruteforceTPPTriples(self, sizes: Tuple[int, int, int], max_search: int = 1000) -> List[Tuple[Set, Set, Set]]:
    """Brute force find TPP triples. Input from Table 1 of (Hedtke & Murthy, 2012)"""
    e = self.identity()
    allElements = set(self.elements())
    otherElements = allElements - {e}
    results = []
    #Generate S subsets of size sizes[0] containing identity
    #Need sizes[0]-1 more elements
    s_combos = itertools.combinations(otherElements, sizes[0] - 1)
    count = 0
    for s_combo in s_combos:
      S_set = {e} | set(s_combo)
      remaining = otherElements - S_set
      #Generate T subsets of size sizes[1] containing identity
      for t_combo in itertools.combinations(remaining, sizes[1] - 1):
        T_set = {e} | set(t_combo)
        remaining2 = remaining - T_set
        #Generate U subsets of size sizes[2] containing identity
        for u_combo in itertools.combinations(remaining2, sizes[2] - 1):
          U_set = {e} | set(u_combo)
          #Quick pre-check: size product > |G| (Observation 2.6)
          if len(S_set) * len(T_set) * len(U_set) <= len(allElements):
            continue
          if self.TestTPPNaive(S_set, T_set, U_set):
              results.append((S_set, T_set, U_set))
          count += 1
          if count >= max_search:
            return results
    return results

## Constructing Our First Group
This is [Section 2.4](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).

We saw in [Section 2.1](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication) that a *cyclic group over the integers* is defined by applying the modulo `%` operator in Python.

(Cohn & Umans, 2003) provide us the product of three cyclic groups as a starting point for our FFT-valid group search.
<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/2013cdb0-725c-4ed4-9586-0793f8c11020_678x174.jpg?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>

Note however, this is an abelian group, so as shown in [Section 2.3](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication) , we won't find useful convolutions.




In [2]:
from dataclasses import dataclass

class CnxCnxCn(Group):
  """The abelian group Cn × Cn × Cn."""
  @dataclass(frozen=True)
  class Element:
    x: int
    y: int
    z: int

    def __repr__(self):
      return f"({self.x},{self.y},{self.z})"
  def __init__(self, modulo):
      self.modulo = modulo
  def elements(self):
    return [
      self.Element(i, j, k)
      for i in range(self.modulo)
      for j in range(self.modulo)
      for k in range(self.modulo)
    ]

  def identity(self):
    return self.Element(0, 0, 0)

  def groupOperation(self, a, b):
    """(a^i b^j c^k)(a^r b^s c^t)= a^(i+r) b^(j+s) c^(k+t)"""
    return self.Element(
      (a.x + b.x) % self.modulo,
      (a.y + b.y) % self.modulo,
      (a.z + b.z) % self.modulo,
    )

  def inverse(self, a):
    """(a^i b^j c^k)^(-1)= a^(-i) b^(-j) c^(-k)"""
    return self.Element(
      (-a.x) % self.modulo,
      (-a.y) % self.modulo,
      (-a.z) % self.modulo,
    )

  def elementToString(self, elem):
    return repr(elem)

def TestCnxCnxCnConstruct():
    G = CnxCnxCn(5)
    e = G.identity()

    #We'll construct our STU because this is a toy example
    S = {G.Element(i, 0, 0) for i in range(G.modulo)}
    T = {G.Element(0, j, 0) for j in range(G.modulo)}
    U = {G.Element(0, 0, k) for k in range(G.modulo)}

    print(f"Order: {len(G.elements())}")
    print(f"Identity: {e}")

    print("\nAll elements:")
    for elem in G.elements():
        print(f"  {elem}", end="")
        if elem == e:
            print("  ← identity", end="")
        print()

    print(f"S = {sorted(S, key=repr)}")
    print(f"T = {sorted(T, key=repr)}")
    print(f"U = {sorted(U, key=repr)}")
    print(f"|S||T||U| = {len(S)*len(T)*len(U)}")
    print("nTesting Triple Product Property:", G.TestTPPNaive(S, T, U))

if __name__ == "__main__":
  TestCnxCnxCnConstruct()

Order: 125
Identity: (0,0,0)

All elements:
  (0,0,0)  ← identity
  (0,0,1)
  (0,0,2)
  (0,0,3)
  (0,0,4)
  (0,1,0)
  (0,1,1)
  (0,1,2)
  (0,1,3)
  (0,1,4)
  (0,2,0)
  (0,2,1)
  (0,2,2)
  (0,2,3)
  (0,2,4)
  (0,3,0)
  (0,3,1)
  (0,3,2)
  (0,3,3)
  (0,3,4)
  (0,4,0)
  (0,4,1)
  (0,4,2)
  (0,4,3)
  (0,4,4)
  (1,0,0)
  (1,0,1)
  (1,0,2)
  (1,0,3)
  (1,0,4)
  (1,1,0)
  (1,1,1)
  (1,1,2)
  (1,1,3)
  (1,1,4)
  (1,2,0)
  (1,2,1)
  (1,2,2)
  (1,2,3)
  (1,2,4)
  (1,3,0)
  (1,3,1)
  (1,3,2)
  (1,3,3)
  (1,3,4)
  (1,4,0)
  (1,4,1)
  (1,4,2)
  (1,4,3)
  (1,4,4)
  (2,0,0)
  (2,0,1)
  (2,0,2)
  (2,0,3)
  (2,0,4)
  (2,1,0)
  (2,1,1)
  (2,1,2)
  (2,1,3)
  (2,1,4)
  (2,2,0)
  (2,2,1)
  (2,2,2)
  (2,2,3)
  (2,2,4)
  (2,3,0)
  (2,3,1)
  (2,3,2)
  (2,3,3)
  (2,3,4)
  (2,4,0)
  (2,4,1)
  (2,4,2)
  (2,4,3)
  (2,4,4)
  (3,0,0)
  (3,0,1)
  (3,0,2)
  (3,0,3)
  (3,0,4)
  (3,1,0)
  (3,1,1)
  (3,1,2)
  (3,1,3)
  (3,1,4)
  (3,2,0)
  (3,2,1)
  (3,2,2)
  (3,2,3)
  (3,2,4)
  (3,3,0)
  (3,3,1)
  (3,3,2)
  (3,3,3)
  (3

## Turning a Matmul into a Discrete Fourier Transform-Style Convolution

This is [Section 3.0](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).
Remember our northstar:
1. We can embed our matrices into a group algebra [Section 3.1](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).
2. Implicitly perform the matmul via convolutions within the group algebra Section [Section 3.2](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).
3. Easily read the matrix product's results from from the group algebra [Section 3.1](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).


---
For the curious, observe that we only count 1 TPP operation during the convolution.
<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/DFT.png?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>

In [3]:
class TPPMatrixMultiplier:
  def __init__(self, group, S, T, U):
    self.group = group
    self.S = list(S)
    self.T = list(T)
    self.U = list(U)

  def Embed(self, A, sLeft, sRight):
    """Embed matrix A into group algebra"""
    out = defaultdict(int)
    for i, s in enumerate(sLeft):
      for j, t in enumerate(sRight):
        g = self.group.groupOperation(self.group.inverse(s), t)
        out[g] += A[i][j]
    return out

  def Extract(self, Z, sLeft, sRight):
    """Extract matrix from group algebra"""
    n1, n2 = len(sLeft), len(sRight)
    M = [[0] * n2 for _ in range(n1)]
    for i, s in enumerate(sLeft):
      for j, t in enumerate(sRight):
        g = self.group.groupOperation(self.group.inverse(s), t)
        M[i][j] = Z.get(g, 0)
    return M

  def ConvolutionMultiply(self, A_hat, B_hat,showConvolution=False):
    """Multiply in the group algebra fft-style"""
    global TPP_OPS
    C = defaultdict(int)
    if(showConvolution == True):
        print(f"TPP Convolutions:")
    for g1, v1 in A_hat.items():
      if(showConvolution == True):
        print(f"{g1}, {v1}")
      for g2, v2 in B_hat.items():
        TPP_OPS += 1
        groupResult = self.group.groupOperation(g1, g2)
        if(showConvolution == True):
          print(f"\t{g2}, {v2}")
        C[groupResult] += v1 * v2
    return C

  def NaiveMatmul(self, A, B):
    """Cubic matmul"""
    global NAIVE_OPS
    n = len(A)
    m = len(B)
    p = len(B[0]) if m > 0 else 0

    C = [[0] * p for _ in range(n)]
    for i in range(n):
      for j in range(p):
        for k in range(m):
          C[i][j] += A[i][k] * B[k][j]
          NAIVE_OPS += 1
    return C

  def TPPMatmul(self, A, B, showConvolution=False):
    """TPP matrix multiplication"""
    A_hat = self.Embed(A, self.S, self.T)
    B_hat = self.Embed(B, self.T, self.U)
    C_hat = self.ConvolutionMultiply(A_hat, B_hat,showConvolution)
    return self.Extract(C_hat, self.S, self.U)

  def CompareMethods_Naive_Convolution(self, A, B, verbose=True, showConvolution = False):
    """Compare naive, TPP"""
    global NAIVE_OPS, TPP_OPS, REDUCED_RANK_OPS
    if verbose:
      print("Comparing Naive vs Convolution)")
      print(f"Matrix A: {len(A)} × {len(A[0]) if A else 0}")
      print(f"Matrix B: {len(B)} × {len(B[0]) if B else 0}")
      print(f"Result:   {len(A)} × {len(B[0]) if B else 0}")
      #Reset counters
      NAIVE_OPS = 0
      TPP_OPS = 0
      REDUCED_RANK_OPS = 0
      #Naive multiplication
      start = time.time()
      cNaive = self.NaiveMatmul(A, B)
      naiveTime = time.time() - start

      #TPP multiplication
      start = time.time()
      cTPP = self.TPPMatmul(A, B, showConvolution)
      tppTime = time.time() - start

      if verbose:
        print("\n" + "-" * 60)
        print("Results:")
        print("-" * 60)
        print(f"Naive multiplication:")
        print(f"  Time: {naiveTime:.6f} seconds")
        print(f"  Mul Operations: {NAIVE_OPS:,}")
        print(f"  cNaive    : {cNaive}")

        print(f"\nTPP multiplication:")
        print(f"  Time: {tppTime:.6f} seconds")
        print(f"  Mul Operations: {TPP_OPS:,}")
        print(f"  cTPP      : {cTPP}")
        print(f"  Matches naive: {cTPP == cNaive}")
    return cNaive, cTPP



## Convolution Test
Observe how we:
1. Embed the left matrix using `S` and `T`.
2. Embed the right matrix using `T` and `U`.
3. Convolve and read the matrix product coefficients, all without performing a matmul.

Our FFT analog yields matching results to the naive matmul! Albeit, the convolution step is rather slow.

**NOTE**: For the curious, you can toggle the showConvolution variable to see the actual calculations:

In [4]:
def TestMulConvolution_CnxCnxCn():
  #Abelian groups are proven to perform worse btw
  G = CnxCnxCn(5)
  e = G.identity()
  #We'll construct our STU because this is a toy example
  S = {G.Element(i, 0, 0) for i in range(G.modulo)}
  T = {G.Element(0, j, 0) for j in range(G.modulo)}
  U = {G.Element(0, 0, k) for k in range(G.modulo)}
  nRows = len(S)
  nMid = len(T)
  nCols = len(U)

  np.random.seed(42)
  maxMatrixValue = 6
  A_rand = np.random.randint(0, maxMatrixValue, size=(nRows, nMid)).tolist()
  B_rand = np.random.randint(0, maxMatrixValue, size=(nMid, nCols)).tolist()
  multiplier = TPPMatrixMultiplier(G, S, T, U)

  print(S)
  print("\nA (random):")
  for row in A_rand:
    print(f"  {row}")
  print("\nB (random):")
  for row in B_rand:
    print(f"  {row}")

  cNaive2, cTPP2 = multiplier.CompareMethods_Naive_Convolution(A_rand, B_rand, verbose=True, showConvolution=True)


if __name__ == "__main__":
    TestMulConvolution_CnxCnxCn()

{(3,0,0), (0,0,0), (4,0,0), (1,0,0), (2,0,0)}

A (random):
  [3, 4, 2, 4, 4]
  [1, 2, 2, 2, 4]
  [3, 2, 5, 4, 1]
  [3, 5, 5, 1, 3]
  [4, 0, 3, 1, 5]

B (random):
  [4, 3, 0, 0, 2]
  [2, 1, 3, 3, 5]
  [5, 5, 2, 3, 3]
  [0, 2, 4, 2, 4]
  [0, 1, 3, 0, 3]
Comparing Naive vs Convolution)
Matrix A: 5 × 5
Matrix B: 5 × 5
Result:   5 × 5
TPP Convolutions:
(2,1,0), 3
	(0,4,4), 4
	(0,4,3), 3
	(0,4,0), 0
	(0,4,2), 0
	(0,4,1), 2
	(0,3,4), 2
	(0,3,3), 1
	(0,3,0), 3
	(0,3,2), 3
	(0,3,1), 5
	(0,0,4), 5
	(0,0,3), 5
	(0,0,0), 2
	(0,0,2), 3
	(0,0,1), 3
	(0,2,4), 0
	(0,2,3), 2
	(0,2,0), 4
	(0,2,2), 2
	(0,2,1), 4
	(0,1,4), 0
	(0,1,3), 1
	(0,1,0), 3
	(0,1,2), 0
	(0,1,1), 3
(2,2,0), 4
	(0,4,4), 4
	(0,4,3), 3
	(0,4,0), 0
	(0,4,2), 0
	(0,4,1), 2
	(0,3,4), 2
	(0,3,3), 1
	(0,3,0), 3
	(0,3,2), 3
	(0,3,1), 5
	(0,0,4), 5
	(0,0,3), 5
	(0,0,0), 2
	(0,0,2), 3
	(0,0,1), 3
	(0,2,4), 0
	(0,2,3), 2
	(0,2,0), 4
	(0,2,2), 2
	(0,2,1), 4
	(0,1,4), 0
	(0,1,3), 1
	(0,1,0), 3
	(0,1,2), 0
	(0,1,1), 3
(2,0,0), 2
	(0,4,4), 4
	(0,4

# How Exactly Do You Find an FFT for the Convolution’s DFT?

No one really knows tbh. Or maybe I do? Lol. If you’re a [LeetArxiv supporter](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication) please reach out and I’ll share the extended subscriber-only version!

# Non-Abelian Example
This is [Section 5.0](https://leetarxiv.substack.com/p/triple-product-property-matrix-multiplication).

Table 1 of (Hedtke & Murthy, 2012) gives us non-abelian groups that satisy the TPP property.

<div align="center">
  <img src="https://github.com/MurageKibicho/The-Annotated-Triple-Product-Property-Matrix-Multiplication-Algorithm/blob/main/Promo/Table.png?raw=true"
       alt="Dua Lipa alongside TPP paper"
       width="600">
</div>

We'll code the  C3 ⋊ C4 example. This represents the semidirect product of the cyclic group of order 3 by the cyclic group of order 4.

We

In [5]:
class C3rtC4(Group):
  """C3 ⋊ C4: a^3 = 1, b^4 = 1, b a b^{-1} = a^{-1}"""
  @dataclass
  class Element:
    a: int  # 0, 1, 2
    b: int  # 0, 1, 2, 3

    def __hash__(self):
      return hash((self.a, self.b))

    def __eq__(self, other):
      return self.a == other.a and self.b == other.b

    def __repr__(self):
      if self.a == 0 and self.b == 0:
        return "1"
      parts = []
      if self.a == 1:
        parts.append("a")
      elif self.a == 2:
        parts.append("a²")
      if self.b == 1:
        parts.append("b")
      elif self.b == 2:
        parts.append("b²")
      elif self.b == 3:
        parts.append("b³")
      return "".join(parts) if parts else "1"
  def elements(self):
    return [self.Element(a, b) for a in range(3) for b in range(4)]
  def identity(self):
    return self.Element(0, 0)
  def groupOperation(self, a, b):
    """(a^i b^j) * (a^k b^l) = a^(i + k*(-1)^j) * b^(j+l)"""
    new_a = (a.a + b.a * ((-1) ** a.b)) % 3
    new_b = (a.b + b.b) % 4
    return self.Element(new_a, new_b)
  def inverse(self, a):
    """(a^i b^j)^(-1) = a^(-i * (-1)^j) * b^(-j)"""
    new_a = (-a.a * ((-1) ** a.b)) % 3
    new_b = (-a.b) % 4
    return self.Element(new_a, new_b)
  def elementToString(self, elem):
    if elem.a == 0 and elem.b == 0:
      return "1"
    parts = []
    if elem.a == 1:
      parts.append("a")
    elif elem.a == 2:
      parts.append("a²")
    if elem.b == 1:
      parts.append("b")
    elif elem.b == 2:
      parts.append("b²")
    elif elem.b == 3:
      parts.append("b³")
    return "".join(parts) if parts else "1"

def TestC3_C4():
  G = C3rtC4()
  e = G.identity()
  a = G.Element(1, 0)
  a2 = G.Element(2, 0)
  b = G.Element(0, 1)
  b2 = G.Element(0, 2)
  b3 = G.Element(0, 3)
  ab = G.Element(1, 1)
  a2b = G.Element(2, 1)
  print("=" * 70)
  print("Testing C3 ⋊ C4 for TPP Triples")
  print("=" * 70)
  print(f"Order: {len(G.elements())}")
  print(f"Identity: {e}")
  print("\nAll elements:")
  for elem in G.elements():
    print(f"  {elem}", end="")
    if elem == e:
      print("  ← identity", end="")
    print()

  print("Brute Force Search for (4,2,2) TPP Triples")
  results = G.BruteforceTPPTriples((4, 2, 2), max_search=5000)
  if results:
    print(f"\nFound {len(results)} TPP triples")
    for i, (S_found, T_found, U_found) in enumerate(results[:5]):
      print(f"\n  Triple {i}:")
      print(f"    S = {[str(x) for x in S_found]}")
      print(f"    T = {[str(x) for x in T_found]}")
      print(f"    U = {[str(x) for x in U_found]}")
      print(f"    Product: {len(S_found)*len(T_found)*len(U_found)}")
  else:
    print("\n❌ No TPP triples found.")

if __name__ == "__main__":
  TestC3_C4()

Testing C3 ⋊ C4 for TPP Triples
Order: 12
Identity: 1

All elements:
  1  ← identity
  b
  b²
  b³
  a
  ab
  ab²
  ab³
  a²
  a²b
  a²b²
  a²b³
Brute Force Search for (4,2,2) TPP Triples

Found 8 TPP triples

  Triple 0:
    S = ['b', 'b²', 'b³', '1']
    T = ['a²b', '1']
    U = ['ab', '1']
    Product: 16

  Triple 1:
    S = ['b', 'b²', 'b³', '1']
    T = ['a²b', '1']
    U = ['ab³', '1']
    Product: 16

  Triple 2:
    S = ['b', 'b²', 'b³', '1']
    T = ['ab', '1']
    U = ['a²b', '1']
    Product: 16

  Triple 3:
    S = ['b', 'b²', 'b³', '1']
    T = ['ab', '1']
    U = ['a²b³', '1']
    Product: 16

  Triple 4:
    S = ['b', 'b²', 'b³', '1']
    T = ['a²b³', '1']
    U = ['ab', '1']
    Product: 16


In [7]:
def TestMul_C3_C4():
  G = C3rtC4()
  e = G.identity()
  a = G.Element(1, 0)
  a2 = G.Element(2, 0)
  b = G.Element(0, 1)
  b2 = G.Element(0, 2)
  b3 = G.Element(0, 3)
  ab = G.Element(1, 1)
  a2b = G.Element(2, 1)
  #Use known working triple 0 from before
  S = {e, b, b2, b3}
  T = {e, a2b}
  U = {e, ab}
  nRows = len(S)
  nMid = len(T)
  nCols = len(U)

  np.random.seed(421)
  maxMatrixValue = 6
  A_rand = np.random.randint(0, maxMatrixValue, size=(nRows, nMid)).tolist()
  B_rand = np.random.randint(0, maxMatrixValue, size=(nMid, nCols)).tolist()
  multiplier = TPPMatrixMultiplier(G, S, T, U)

  print("\nA (random):")
  for row in A_rand:
    print(f"  {row}")
  print("\nB (random):")
  for row in B_rand:
    print(f"  {row}")

  cNaive2, cTPP2 = multiplier.CompareMethods_Naive_Convolution(A_rand, B_rand, verbose=True)


if __name__ == "__main__":
    TestMul_C3_C4()


A (random):
  [5, 4]
  [3, 1]
  [4, 1]
  [4, 0]

B (random):
  [3, 5]
  [0, 0]
Comparing Naive vs Convolution)
Matrix A: 4 × 2
Matrix B: 2 × 2
Result:   4 × 2

------------------------------------------------------------
Results:
------------------------------------------------------------
Naive multiplication:
  Time: 0.000012 seconds
  Mul Operations: 16
  cNaive    : [[15, 25], [9, 15], [12, 20], [12, 20]]

TPP multiplication:
  Time: 0.000164 seconds
  Mul Operations: 32
  cTPP      : [[15, 25], [9, 15], [12, 20], [12, 20]]
  Matches naive: True


Thanks for making it this far! Here are some  [free GPU credits](https://runpod.io?ref=0wy3bt8r) if you’d like to try this on Runpod (first come, first serve).

The LeetArxiv Substack shows programmers how to turn advanced math papers into code. You might also like:
1. [Why Compiler Engineers Rarely Use Strassen's Algorithm for Fast Matrix Multiplications](https://leetarxiv.substack.com/p/why-compilers-rarely-use-strassens-algorithm)
2. [This Secret Math Equation let the US Government Spy on Anyone](https://leetarxiv.substack.com/p/dual-ec-backdoor-coding-guide)
3. [If you're smart why are you poor? Elliptic Curve Edition](https://leetarxiv.substack.com/p/if-youre-smart-why-are-you-poor-elliptic)


